In [1]:
import os
os.chdir('../../..')

In [2]:
import polars as pl

from src.helper_functions import create_chemiscope_viewer, average_numeric_by_cluster
from src.datasets import QM9Dataset
from src.outlier_detection import hdbscan_outliers, ocsvm_outliers, lof_outliers

INFO: Enabling RDKit 2025.09.4 jupyter extensions


In [3]:
descriptor = "soap"
outliers = pl.read_parquet('data/QM9/outliers/synthetic_outliers.parquet')

In [4]:
qm9 = QM9Dataset(limit=5000, 
                 sampling_strategy="stratified",
                 stratify_by=["num_atoms", "gap"], 
                 injected_molecules = outliers,
                 descriptors=[descriptor],
                 )
df = qm9.load()

2026-04-23 16:00:28.963 | INFO     | src.datasets:load:796 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-23 16:00:29.162 | INFO     | src.datasets:_sample_qm9_df:998 - QM9 sampling complete: strategy=stratified, requested_limit=5500, returned_rows=5500.
2026-04-23 16:00:31.524 | INFO     | src.datasets:_add_requested_descriptors:188 - Applying requested QM9 descriptors to sampled dataframe (rows=5519).
2026-04-23 16:00:31.847 | INFO     | src.features:compute_soap:176 - Computing SOAP (rcut=6.0, nmax=8, lmax=6)...
2026-04-23 16:01:07.371 | SUCCESS  | src.datasets:add_soap:1150 - Added SOAP embeddings.
2026-04-23 16:01:07.372 | INFO     | src.datasets:_add_requested_descriptors:213 - Added descriptor column(s): ['soap_embedding']
2026-04-23 16:01:07.372 | WARNING  | src.datasets:inject_outliers:779 - Some injected SMILES were skipped because parsing or 3D embedding failed: ['C1#CC#CC#C1']
2026-04-23 16:01:07.372 | INFO     | src.datasets:inject_outlier

In [5]:
df.filter(pl.col("is_injected") == 1)

mol_id,formula,smiles,canonical_smiles,scaffold_smiles,generic_scaffold,root_scaffold,brics_fragments,scaffold_tree_nodes,selfies,functional_groups,structure_class,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,num_fluorine,num_heteroatoms,num_atoms,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,is_injected,outlier_category,soap_embedding
str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,list[f64]
"""injected_0""","""C12H10""","""c1ccccc1-c2ccccc2""","""[H]c1c([H])c([H])c(-c2c([H])c(…","""c1ccc(-c2ccccc2)cc1""","""C1CCC(C2CCCCC2)CC1""","""*1:*:*:*:*:*:1""","""[16*]c1c([H])c([H])c([H])c([H]…","""[H]c1c([H])c([H])c(-c2c([H])c(…","""[H][C][=C][Branch1][C][H][C][B…","""benzene""","""Aromatic""",154,3,0,1.031244,12.323082,12,2,2,0,0,22,2.090909,1,0.0,1.0,0.0,0,0,12,0,12,0,9,44,1.259914,2,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,"""size_outliers""","[0.088123, 0.241011, … 0.0]"
"""injected_1""","""C9H8O4""","""CC(=O)Oc1ccccc1C(=O)O""","""[H]OC(=O)c1c([H])c([H])c([H])c…","""c1ccccc1""","""C1CCCCC1""","""*1:*:*:*:*:*:1""","""[1*]C(=O)C([H])([H])[H],[3*]O[…","""[H]OC(=O)c1c([H])c([H])c([H])c…","""[H][O][C][=Branch1][C][=O][C][…","""benzene,carboxylic_acid,ester,…","""Aromatic""",180,1,63,1.106528,12.600109,13,1,1,0,4,21,2.0,3,0.0,0.888889,0.111111,1,4,9,0,8,1,8,39,1.268438,1,0,0,0,0,1,1,0,1,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,"""size_outliers""","[0.074392, 0.20155, … 0.0]"
"""injected_2""","""C13H18O2""","""CC(C)Cc1ccc(cc1)C(C)C(=O)O""","""[H]OC(=O)C([H])(c1c([H])c([H])…","""c1ccccc1""","""C1CCCCC1""","""*1:*:*:*:*:*:1""","""[1*]C(=O)C([8*])([H])C([H])([H…","""[H]OC(=O)C([H])(c1c([H])c([H])…","""[H][O][C][=Branch1][C][=O][C][…","""benzene,carboxylic_acid""","""Aromatic""",206,3,37,0.99713,12.678536,15,1,1,0,2,33,2.0,7,0.0,0.538462,0.461538,1,2,13,0,7,6,11,72,1.250021,1,0,0,0,0,1,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,"""size_outliers""","[0.105975, 0.290581, … 0.0]"
"""injected_3""","""C10H16""","""C1C2CC3CC1CC(C2)C3""","""[H]C1([H])C2([H])C([H])([H])C3…","""C1C2CC3CC1CC(C2)C3""","""C1C2CC3CC1CC(C2)C3""","""*1*2**3**1**(*2)*3""","""[H]C1([H])C2([H])C([H])([H])C3…","""[H]C1([H])C2([H])C([H])([H])C3…","""[H][C][Branch1][C][H][C][Branc…","""""","""Aliphatic Ring""",136,2,0,0.94955,12.699147,10,4,0,0,0,26,2.153846,0,0.0,0.0,1.0,0,0,10,0,0,10,6,61,1.276441,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,"""size_outliers""","[0.117899, 0.328894, … 0.0]"
"""injected_4""","""C9H10N2O4""","""O=C1CCC(=O)N1C2CCC(=O)NC2=O""","""[H]N1C(=O)C([H])([H])C([H])([H…","""O=C1CCC(N2C(=O)CCC2=O)C(=O)N1""","""CC1CCC(C2C(C)CCC2C)C(C)C1""","""*=*1***(=*)*1""","""[15*]C1([H])C(=O)N([H])C(=O)C(…","""[H]N1C(=O)C([H])([H])C([H])([H…","""[H][N][C][=Branch1][C][=O][C][…","""amide""","""Aliphatic Ring""",210,-1,83,0.877819,12.834697,15,2,0,0,6,25,2.08,1,0.0,0.444444,0.555556,1,4,11,0,4,5,8,55,1.297727,0,0,0,0,4,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,"""size_outliers""","[0.077527, 0.210961, … 0.0]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

In [6]:
if descriptor == "soap":
    dist_matrix = qm9.get_distance_matrix(descriptor, "soap_kernel",force_calculate=True)
else:
    dist_matrix = qm9.get_distance_matrix(descriptor, "euclidean",force_calculate=True)

2026-04-23 16:01:11.414 | INFO     | src.datasets:get_distance_matrix:1342 - Calculating distance matrix for soap using soap_kernel distance.
2026-04-23 16:01:12.860 | SUCCESS  | src.distance:_compute_and_save:74 - Saved distance matrix to data/QM9/dist_soap_soap_kernel.npy


In [7]:
df = hdbscan_outliers(df, dist_matrix)
df = ocsvm_outliers(df, dist_matrix)
df = lof_outliers(df, dist_matrix)

/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


HDBSCAN — 6 distinct: -1: 2929, 0: 204, 1: 336, 2: 1148, 3: 191, 4: 211
OCSVM — 2 distinct: -1: 502, 1: 4517
LOF — 2 distinct: -1: 33, 1: 4986


In [8]:
df = df.with_columns([
    (pl.col("hdbscan_labels") == -1).cast(pl.Int8).alias("is_outlier_hdbscan"),
    (pl.col("ocsvm_label") == -1).cast(pl.Int8).alias("is_outlier_ocsvm"),
    (pl.col("lof_label") == -1).cast(pl.Int8).alias("is_outlier_lof"),
])

# 2. Create a "consensus score" (0 to 3)
df = df.with_columns(
    (pl.col("is_outlier_hdbscan") + 
        pl.col("is_outlier_ocsvm") + 
        pl.col("is_outlier_lof")).alias("outlier_votes")
)

# 3. Analyze the overlap
vote_counts = df["outlier_votes"].value_counts().sort("outlier_votes")
print("\n--- Outlier Consensus Summary ---")
print(vote_counts)

# 4. Extract the exact overlapping groups
overlap_all_three = df.filter(pl.col("outlier_votes") == 3)
overlap_at_least_two = df.filter(pl.col("outlier_votes") >= 2)

df = df.with_row_index("original_index")
consensus_indices = df.filter(pl.col("outlier_votes") == 3)["original_index"].to_list()


--- Outlier Consensus Summary ---
shape: (4, 2)
┌───────────────┬───────┐
│ outlier_votes ┆ count │
│ ---           ┆ ---   │
│ i8            ┆ u32   │
╞═══════════════╪═══════╡
│ 0             ┆ 1933  │
│ 1             ┆ 2736  │
│ 2             ┆ 322   │
│ 3             ┆ 28    │
└───────────────┴───────┘


In [9]:
create_chemiscope_viewer(df, dist_matrix, df["outlier_votes"].to_numpy(), 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [10]:
average_numeric_by_cluster(df, "outlier_votes")

shape: (4, 80)
┌─────────────┬───────┬─────────────┬─────────────┬────────────┬─────────┬─────────┬─────────────┬─────────────┬─────────────┬───────────┬─────────────┬─────────────┬─────────────┬───────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬─────────────┬──────────┬───────────┬──────────┬──────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬─────────┬────────┬────────┬─────────────┬─────────────┬────────────┬────────────┬───────────┬───────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────

outlier_votes,count,token_to_atom_ratio,original_index,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,num_fluorine,num_heteroatoms,num_atoms,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,…,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,is_injected,hdbscan_labels,ocsvm_label,ocsvm_score,lof_label,lof_score,is_outlier_hdbscan,is_outlier_ocsvm,is_outlier_lof,pct_aliphatic_ring,pct_aromatic,pct_acyclic,avg_len_selfies,unique_scaffolds,top_scaffold,top_scaffold_pct,unique_generic_scaffolds,top_generic_scaffold,top_generic_scaffold_pct,unique_outlier_categories,top_outlier_category,top_outlier_category_pct
i8,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f64,f64,f64,f64,f64,f64,f64,u32,str,f64,u32,str,f64,u32,str,f64
0,1933,2.152199,3130.345059,125.403518,0.133989,29.130367,0.903782,12.826318,8.953958,2.006208,0.010347,0.0,1.993275,20.247801,2.089085,2.425246,0.026754,0.113541,0.859705,0.899638,1.629591,7.277289,0.186239,0.777031,5.997413,6.327988,43.642525,1.263818,0.0,0.414899,0.002069,0.069322,0.106053,0.001552,0.017072,0.111226,…,7.288755,1206.447856,4.626968,-11106.509421,-11106.271975,-11106.246274,-11107.421151,33.223475,-82.725936,-83.255531,-83.741376,-76.896143,2.975433,1.364824,1.118731,0.0,1.940507,1.0,11.928772,1.0,1.034343,0.0,0.0,0.0,94.361097,1.034661,4.604242,43.642525,832,"""Acyclic""",4.604242,228,"""C1CC1""",8.329022,1,null,0.0
1,2736,2.02875,2179.275219,121.255117,0.062865,38.497442,0.870268,12.829843,8.690424,1.497807,0.141813,0.001096,2.465643,17.753655,2.048974,2.289474,0.092669,0.231696,0.675635,0.934942,2.075292,6.217105,0.610746,1.360015,4.25402,6.365132,36.229532,1.261467,0.002924,0.351243,0.031798,0.162646,0.149854,0.002924,0.051535,0.146199,…,6.740859,1185.69853,3.901264,-11097.053126,-11096.821303,-11096.79561,-11097.965846,31.41718,-74.382513,-74.825389,-75.249624,-69.270221,3.46936,1.406408,1.119836,0.000365,-0.833699,0.885234,11.670557,1.0,1.075359,0.942617,0.057383,0.0,72.112573,13.998538,13.888889,36.229532,827,"""Acyclic""",13.888889,222,"""Acyclic""",13.888889,2,"""extreme_outliers""",0.03655
2,322,1.822156,1558.39441,118.596273,-0.121118,52.822981,0.753925,12.89009,8.509317,1.083851,0.481366,0.015528,3.354037,14.795031,2.006497,1.751553,0.132632,0.495931,0.371437,1.009317,2.819876,5.099379,0.791925,2.273292,2.090062,6.23913,27.590062,1.261781,0.009317,0.28882,0.080745,0.263975,0.136646,0.006211,0.099379,0.149068,…,6.023631,1112.431894,2.982626,-11306.775335,-11306.561547,-11306.535897,-11307.669215,28.331256,-63.64702,-63.990318,-64.336084,-59.414723,4.458865,1.516011,1.152851,0.015528,-1.0,-0.968944,-15.78626,0.968944,1.155042,1.0,0.984472,0.015528,33.540373,44.409938,22.049689,27.590062,154,"""Acyclic""",22.049689,55,"""C1CCCC1""",26.086957,4,"""extreme_outliers""",0.621118
3,28,1.799181,2764.785714,113.821429,0.464286,32.071429,0.9702,13.01005,7.321429,0.928571,0.464286,0.535714,2.821429,12.714286,1.933162,1.285714,0.119048,0.391865,0.417659,0.571429,1.892857,4.535714,0.464286,2.321429,1.714286,5.035714,24.142857,1.289719,0.214286,0.107143,0.035714,0.107143,0.0,0.071429,0.035714,0.035714,…,6.568829,782.883034,2.187685,-11034.254036,-11034.074805,-11034.049316,-11035.09694,23.009934,-49.101313,-49.361545,-49.628803,-45.839429,11.879523,2.228035,1.853031,0.464286,-1.0,-1.0,-72.640628,-1.0,2.177519,1.0,1.0,1.0,21.428571,35.714286,42.857143,24.142857,17,"""Acyclic""",42.857143,11,"""Acyclic""",42.857143,5,"""element_outliers""",17.857143
